# 01. Подготовка данных

Локальный pipeline скачивает четыре публичных архива, материализует только необходимые изображения и кропы и создаёт splits. NII не используется. Все операции записи выключены по умолчанию.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()), None)
assert PROJECT_ROOT is not None, 'Откройте notebook из клонированного DiatomDINO'
os.chdir(PROJECT_ROOT)
CONFIG = PROJECT_ROOT / 'configs/data.yaml'
DATA_ROOT = PROJECT_ROOT / 'data'
RUN_ENVIRONMENT_CHECK = False
RUN_DRY_RUN = False
RUN_DOWNLOAD_BUILD_SPLIT = False
print('Project:', PROJECT_ROOT.resolve())
print('Data root:', DATA_ROOT.resolve())

## Конфигурация источников

In [ ]:
from core.config_loader import load_config
config = load_config(CONFIG)
print('Sources:', sorted(config['sources']))
assert 'nii' not in {name.lower() for name in config['sources']}
print('Detector split:', config['splits']['detector'])
print('Classifier validation fraction:', config['splits']['classifier_validation'])
print('Gunduz gallery fraction:', config['splits']['benchmark_gallery'])

## Проверка локального окружения
Команда только сообщает версию CUDA, GPU и свободное место; ничего не скачивает.

In [ ]:
if RUN_ENVIRONMENT_CHECK:
    subprocess.run([sys.executable, '-m', 'scripts.check_environment', '--data-root', 'data', '--minimum-free-gb', '100'], check=True)
else:
    print('Environment check skipped')

## План и сборка
Сначала выполните dry-run. Для реальной загрузки и материализации отдельно включите последний флаг.

In [ ]:
base_command = [sys.executable, '-m', 'scripts.prepare_data', 'all', '--config', str(CONFIG)]
if RUN_DRY_RUN:
    subprocess.run(base_command + ['--dry-run'], check=True)
else:
    print('Dry-run skipped')

if RUN_DOWNLOAD_BUILD_SPLIT:
    assert not (DATA_ROOT / 'datasetDiatom').exists(), 'Готовый dataset не перезаписывается: выберите новый data_root'
    subprocess.run(base_command, check=True)
else:
    print('Download/build/split skipped')

## Аудит результата

In [ ]:
audit_files = sorted(DATA_ROOT.glob('**/audit.json'))
for path in audit_files:
    print('\n', path.relative_to(PROJECT_ROOT))
    print(json.dumps(json.loads(path.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
if not audit_files:
    print('Audit files появятся после сборки splits')